# Tutorial

A complete run on a simulated cohort, narrated. Nothing is downloaded and
the whole thing takes about a minute.

The cohort is simulated along a branching tree, so at the end there is
something to check the result against: we know what shape the embedding
should recover.

## 1. A cohort to work with

`manifold-genetics init synthetic` writes everything a run needs — genotypes,
labels, a colormap and a config — into one directory.

In [ ]:
import tempfile
from pathlib import Path

work = Path(tempfile.mkdtemp())
!manifold-genetics init synthetic --out {work}

Those are the four kinds of input the pipeline takes:

In [ ]:
for path in sorted(work.rglob('*')):
    if path.is_file():
        print(f'{path.relative_to(work)}  ({path.stat().st_size:,} bytes)')

The genotypes are PLINK 1 triples. The labels say which branch of the tree
each sample was drawn from:

In [ ]:
import pandas as pd

labels = pd.read_csv(work / 'data' / 'labels.csv')
print(labels['branch'].value_counts().sort_index().to_string())
labels.head()

## 2. What the config says

A run is driven by one YAML file. `--dry-run` prints every setting it
resolves to, marking `(default)` on anything the package supplied rather than
the file — the preset contributes most of the embedding parameters.

In [ ]:
!manifold-genetics run {work}/config.yaml --dry-run

## 3. Run it

PCA, then the embedding, then the figures.

In [ ]:
!manifold-genetics run {work}/config.yaml

Every stage writes the same shape of file: a `sample_id` column followed by
`dim_1 … dim_n`. That is what lets them compose.

In [ ]:
pcs = pd.read_csv(work / 'outputs' / 'pca' / 'project_pca_10.csv')
emb = pd.read_csv(work / 'outputs' / 'embeddings' / 'phate_2d.csv')

print('PCA      ', pcs.shape, list(pcs.columns[:4]))
print('embedding', emb.shape, list(emb.columns))

## 4. Did it work?

This is the reason for simulating along a tree. The left panel is the tree the
cohort was drawn from — the ground truth. The right is what PHATE recovered
from the genotypes alone, having never seen it.

Branches should appear in both, in the same colours. Branches separated by an
unsampled gap edge in the tree should come out detached in the embedding.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

truth = mpimg.imread(work / 'dla_tree_ground_truth.png')
found = mpimg.imread(
    work / 'outputs' / 'figures' / 'embeddings' / 'project_phate_by_branch.png'
)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, image, title in zip(axes, (truth, found), ('Ground truth', 'PHATE embedding')):
    ax.imshow(image)
    ax.set_title(title, fontsize=15)
    ax.axis('off')
plt.tight_layout()

## Where to go next

- [Quickstart](quickstart.md) — the same commands on a real cohort
- [Formats](formats.md) — every file the pipeline reads and writes
- [Command line](cli.md) — running the stages individually